# Digitization Pipeline Attribution — Reference Solution

Recovers which real digitization batch produced each OCR'd newspaper text
snippet, using only the OCR pipeline's character-error fingerprint. Shows
three submissions of increasing sophistication, each graded against the
private answer key to demonstrate the measured skill gradient documented in
`DESIGN.md`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "..")  # repo root, for grade.py
sys.path.insert(0, ".")   # work/, for features.py / char_encoder.py

PUBLIC = Path("./dataset/public")
PRIVATE = Path("./dataset/private")

train = pd.read_csv(PUBLIC / "train.csv")
test = pd.read_csv(PUBLIC / "test.csv")
answers = pd.read_csv(PRIVATE / "answers.csv")

print(f"train: {len(train)} labeled snippets across {train['batch_id'].nunique()} batches")
print(f"test: {len(test)} snippet-rows across {test['bag_id'].nunique()} bags")

train: 4714 labeled snippets across 19 batches
test: 4457 snippet-rows across 200 bags


## Submission 1 — baseline: every snippet its own group

A safe, always-valid starting point. Scores 0.000 by construction (Adjusted
Rand Index chance-corrects this to exactly zero) but proves the pipeline
runs end to end before any real modeling begins.

In [ ]:
from grade import grade

def make_submission(bag_to_labels):
    rows = []
    for bag_id, group in test.groupby("bag_id"):
        labels = bag_to_labels(bag_id, group)
        for sid, lab in zip(group["snippet_id"], labels):
            rows.append({"row_id": f"{bag_id}::{sid}", "bag_id": bag_id, "snippet_id": sid, "group_label": lab})
    return pd.DataFrame(rows)

def score(sub_df):
    return grade(sub_df, answers)

sub_v1 = make_submission(lambda bag_id, group: range(len(group)))
sub_v1.to_csv("./working/submission_v1_baseline.csv", index=False)
print("submission 1 (all-singleton) score:", score(sub_v1))

submission 1 (all-singleton) score: 0.0


## Feature engineering — OCR noise fingerprint features

Hand-crafted, content-free statistics: junk-character rates, letter
confusion-pair frequencies, out-of-vocabulary word ratio, and similar
low-level noise signals. None of these read the actual subject matter of an
article, only how garbled the OCR output is.

In [ ]:
import re
import string

COMMON_ENGLISH = set(
    "the of and to a in that is was he for it with as his on be at by i this had not "
    "are but from or have an they which one you were her all she there would their we "
    "him been has when who will more no if out so said what up its about into than them "
    "can only other new some could time these two may then do first any my now such like "
    "our over man me even most made after also did many before must through back years "
    "where much your way well down should because each just those people mr how too little "
    "state good very make world still see own men work long here get both between city "
    "under never day same another know while last might us great old year come since "
    "against go came right used take three".split()
)
CONFUSION_PAIRS = ["rn", "cl", "li", "ii", "vv", "nn", "ri"]
FEATURE_NAMES = [
    "len", "non_ascii_ratio", "digit_ratio", "punct_ratio", "alpha_ratio",
    "upper_mid_word_ratio", "oov_word_ratio", "single_char_token_ratio",
    "repeated_char_run_rate", "question_mark_rate", "confusion_pair_rate",
    "digit_letter_mix_rate", "avg_word_len", "n_words",
]

def extract_features(text):
    if not text:
        text = ""
    n = len(text)
    words = text.split()
    n_words = max(1, len(words))
    alpha = sum(c.isalpha() for c in text)
    digit = sum(c.isdigit() for c in text)
    punct = sum(c in string.punctuation for c in text)
    non_ascii = sum(ord(c) > 127 for c in text)
    upper_mid_word = sum(1 for w in words if len(w) > 2 and any(c.isupper() for c in w[1:-1]))
    lower_words = [w.strip(string.punctuation).lower() for w in words]
    oov = sum(1 for w in lower_words if w and w.isalpha() and w not in COMMON_ENGLISH)
    single_char_tokens = sum(1 for w in words if len(w) == 1 and w.isalpha())
    repeated_char_runs = len(re.findall(r"(.)\1{2,}", text))
    question_marks = text.count("?")
    confusion_hits = sum(text.count(p) for p in CONFUSION_PAIRS)
    digit_letter_mix = len(re.findall(r"[a-zA-Z]\d|\d[a-zA-Z]", text))
    avg_word_len = sum(len(w) for w in words) / n_words
    return {
        "len": n, "non_ascii_ratio": non_ascii / max(1, n), "digit_ratio": digit / max(1, n),
        "punct_ratio": punct / max(1, n), "alpha_ratio": alpha / max(1, n),
        "upper_mid_word_ratio": upper_mid_word / n_words, "oov_word_ratio": oov / n_words,
        "single_char_token_ratio": single_char_tokens / n_words,
        "repeated_char_run_rate": repeated_char_runs / max(1, n),
        "question_mark_rate": question_marks / max(1, n),
        "confusion_pair_rate": confusion_hits / max(1, n),
        "digit_letter_mix_rate": digit_letter_mix / max(1, n),
        "avg_word_len": avg_word_len, "n_words": n_words,
    }

def vector(text):
    f = extract_features(text)
    return [f[k] for k in FEATURE_NAMES]

print(f"{len(FEATURE_NAMES)} hand-crafted features per snippet")

14 hand-crafted features per snippet


## Submission 2 — unsupervised clustering on hand features (no training)

Cluster each bag's snippets directly on the raw noise-feature vectors, with
no use of the labeled training set at all. This is the "what does zero
learning get you" baseline.

In [ ]:
from sklearn.cluster import AgglomerativeClustering

def guess_k(n_items):
    return max(2, round(n_items / 6))

def unsupervised_labels(bag_id, group):
    vals = np.array([vector(t) for t in group["text"]])
    vals = (vals - vals.mean(0)) / (vals.std(0) + 1e-9)
    k = max(1, min(guess_k(len(group)), len(group)))
    if k == 1:
        return [0] * len(group)
    return AgglomerativeClustering(n_clusters=k).fit_predict(vals)

sub_v2 = make_submission(unsupervised_labels)
sub_v2.to_csv("./working/submission_v2_unsupervised.csv", index=False)
print("submission 2 (unsupervised features) score:", score(sub_v2))

submission 2 (unsupervised features) score: 0.09939338832529877


## Submission 3 (final) — trained RandomForest-embedding reference

Train a RandomForest on the 19 LABELED train batches to predict batch
identity from the same hand-crafted features. For test snippets (from 16
batches the model has never seen), use the predicted probability vector over
the 19 known classes as a learned embedding: two snippets from the same
unseen batch should land on a similar distribution over known fingerprint
"neighbourhoods," even though the exact class is novel. Cluster each bag on
that embedding.

This is the shipped reference (`reference_solution.py`), reproduced here
for the record.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X_train = np.array([vector(t) for t in train["text"]])
y_train = train["batch_id"].values
clf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1)
clf.fit(X_train, y_train)

def trained_labels(bag_id, group):
    vals = np.array([vector(t) for t in group["text"]])
    proba = clf.predict_proba(vals)
    k = max(1, min(guess_k(len(group)), len(group)))
    if k == 1:
        return [0] * len(group)
    return AgglomerativeClustering(n_clusters=k, metric="cosine", linkage="average").fit_predict(proba)

sub_final = make_submission(trained_labels)
Path("./working").mkdir(exist_ok=True)
sub_final.to_csv("./working/submission.csv", index=False)
print("submission 3 (trained RF embedding, FINAL) score:", score(sub_final))

submission 3 (trained RF embedding, FINAL) score: 0.31602306496061555


## Summary

| Submission | Approach | ARI |
|---|---|---|
| 1 | all-singleton baseline | 0.000 |
| 2 | unsupervised feature clustering, no training | 0.099 |
| 3 (final) | trained RandomForest-embedding, per-bag clustering | 0.316 |

Full baseline ladder, adversarial-shortcut checks, and the honest
development history (including two real bugs found and fixed during
construction) are documented in `DESIGN.md`. A from-scratch neural
character-CNN encoder (`work/char_encoder.py`) was also tried and measured
weaker than this RandomForest approach at the current data scale — real
headroom above this reference, not a solved task.